In [19]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn import datasets

In [20]:
# 데이터 로딩

data = datasets.load_wine()

X = pd.DataFrame(data.data, columns = data.feature_names)
Y = pd.DataFrame(data.target, columns = ['label'])

df = pd.concat([X,Y], axis = 1)
df.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,label
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0


In [21]:
#features 순위 가져오기: 적합하고 영향력이 큰 특징 가져옵니다 단, 모델 학습을 돌려야합니다
# 그러니까 미리 x 줄여서 차원축소

# 차원축소
 - 원래 데이터를 합쳐서 새로운 특징을 만듦 (13개에서 줄어들고요)
 - feature: x 특징
 - feature_importance: 기존 특징은 유지, 누가 더 중요한지 체크해줌. (13개 유지, 영향력 높은 피쳐가 뭔지 알기 쉬워용)

 - 단점: 해석이 어려워짐
 - 장점: 데이터 압축, 비용 감소.


In [22]:
#1. 임포트
from sklearn.decomposition import PCA

#2. 객체 생성 -> 비지도, 몇차원으로 줄일건지? n_components =  필수 옵션. (현 특성 수보다 작아야함!)
pca = PCA(n_components=5)

#3. 적용
pca_x = pca.fit_transform(X)
#print(pca_x)        #[ 3.18562979e+02  2.14921307e+01 -3.13073470e+00  2.50113758e-01 -6.77078222e-01]

pca_x = pd.DataFrame(pca_x)
pca_x.head()

,0,1,2,3,4
0,318.562979,21.492131,-3.130735,0.250114,-0.677078
1,303.097420,-5.364718,-6.822835,0.864035,0.486096
2,438.061133,-6.537309,1.113223,-0.912411,-0.380651
3,733.240139,0.192729,0.917257,0.541251,-0.858662
4,-11.571428,18.489995,0.554422,-1.360896,-0.276442


In [23]:
#비교

#kmeans로 13개 특성 군집화 vs. 5개 특성 군집화 => 성능 비교 (실루엣 계수)
from sklearn.cluster import KMeans

#성능비교용으로 군집 개수 3개 통일
km13 = KMeans(n_clusters = 3)
km5 = KMeans(n_clusters=3)

#학습
km13.fit(X)     #원본
km5.fit(pca_x)  #차원축소된 결과

#예측
km13_pred = km13.predict(X)
km5_pred = km5.predict(pca_x)

#평가) 비교
from sklearn.metrics import silhouette_score

km13_s = silhouette_score(X, km13_pred)
km5_s = silhouette_score(pca_x, km5_pred)

print(f'13개 특징 실루엣: {km13_s} vs {km5_s} 5개 특징 실루엣')
#13개 특징 실루엣: 0.5711381937868838 vs 0.5712371867493663 5개 특징 실루엣

#별 차이 읎는디 껄껄껄
#차원(특징)은 적지만 설명력이 좋은(실루엣 계수가 높은)것이 더 효율적

13개 특징 실루엣: 0.5595823478987213 vs 0.5712371867493663 5개 특징 실루엣


##TSNE 알고리즘 이용해 차원 축소 한 후, 실루엣 계수 비교


In [35]:
from sklearn.manifold import TSNE


#차원 3개로 축소, 13개 차원일 때 값과 3개 차원일 때 값 비교
#비교 알고리즘 KMeans

tsne3 = TSNE(n_components = 3)              #13차원을 가진 X(train) 데이터 축소


tsne_x = pd.DataFrame(tsne_x)               #데이터프레임화(이거 필요한가): ndarray여도 가능! 꼭 훈련에 데프만 가능한건 아님 그래서 데프화 해도 되고 안해도 되고~


#비교 -  KMeans
#객체정의
km13 = KMeans(n_clusters = 3)       #13차원
km3 = KMeans(n_clusters = 3)        #3차원


#훈련
#km13에 들어갈 훈련 데이터는 DataFrame 형태의 X(13차원)
km13.fit(X)

#km3에 들어갈 훈련 데이터는 3차원의 tsne_x
#sklearn fit 함수: 입력으로 DataFrame or np.ndarray 모두 가능
tsne_x = tsne3.fit_transform(X)
km3.fit(tsne_x)

#예측
km13_pred = km13.predict(X)
km3_pred = km3.predict(tsne_x)

km13_s = silhouette_score(X, km13_pred)
km3_s = silhouette_score(tsne_x, km3_pred)
                        #{silhouette_score(X, km13_pred)} 이런 형식으로 가능
print(f'13개 특징 실루엣: {km13_s} vs. 3개 축소 특징 실루엣: {km3_s}' )

13개 특징 실루엣: 0.5711381937868838 vs. 3개 축소 특징 실루엣: 0.6109687089920044


In [ ]:
# 이외: Truncated SVD (특이값 분해)
from sklearn.decomposition import TruncatedSVD